In [ ]:
'''Lane Change Violation Detection using Indicator Recognition
This notebook presents a demo-level computer vision pipeline to detect unsafe lane change behavior from traffic video using vehicle tracking and indicator analysis.'''

!pip install -q ultralytics opencv-python filterpy pandas

import cv2
import os
import numpy as np
from collections import deque
from ultralytics import YOLO

# Paths
INPUT_VIDEO = "/inputvideo1.mp4"
OUTPUT_VIDEO = "/content/output_video/result.mp4"

# Create folders
os.makedirs("/content/output_video", exist_ok=True)
os.makedirs("/content/evidence", exist_ok=True)
os.makedirs("/content/challan", exist_ok=True)

print("✅ Paths & folders ready")


In [ ]:
# Load YOLOv8 model
model = YOLO("yolov8n.pt")

print("✅ YOLOv8 model loaded")


In [ ]:
# Store centroid history
lane_history = {}

LANE_WINDOW = 8
LANE_THRESHOLD_RATIO = 0.06  # % of frame width

def update_lane_history(tid, cx):
    if tid not in lane_history:
        lane_history[tid] = deque(maxlen=LANE_WINDOW)
    lane_history[tid].append(cx)


def detect_lane_change(tid, frame_width):
    if tid not in lane_history:
        return {"changed": False}

    history = lane_history[tid]
    if len(history) < LANE_WINDOW:
        return {"changed": False}

    dx = history[-1] - history[0]
    threshold = LANE_THRESHOLD_RATIO * frame_width

    if abs(dx) > threshold:
        direction = "LEFT" if dx < 0 else "RIGHT"
        return {"changed": True, "direction": direction}

    return {"changed": False}


In [ ]:
# HSV range for yellow/orange indicators
IND_LOWER = np.array([5, 120, 120])
IND_UPPER = np.array([35, 255, 255])

IND_PIXEL_THRESHOLD = 0.015
IND_HISTORY = 8

indicator_history = {}

def get_indicator_rois(frame, bbox):
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    y_start = y1 + int(0.7 * h)

    left_roi  = frame[y_start:y2, x1:x1 + int(0.25 * w)]
    right_roi = frame[y_start:y2, x2 - int(0.25 * w):x2]

    return left_roi, right_roi


def indicator_on(roi):
    if roi.size == 0:
        return False

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, IND_LOWER, IND_UPPER)
    ratio = np.sum(mask > 0) / (roi.shape[0] * roi.shape[1] + 1e-6)

    return ratio > IND_PIXEL_THRESHOLD


def update_indicator_history(frame, tracks):
    for trk in tracks:
        x1, y1, x2, y2, tid = map(int, trk)

        if tid not in indicator_history:
            indicator_history[tid] = deque(maxlen=IND_HISTORY)

        left_roi, right_roi = get_indicator_rois(frame, (x1, y1, x2, y2))
        indicator_history[tid].append(
            (indicator_on(left_roi), indicator_on(right_roi))
        )


def indicator_used(tid, direction):
    if tid not in indicator_history:
        return False

    history = indicator_history[tid]

    if direction == "LEFT":
        return any(l for l, r in history)
    if direction == "RIGHT":
        return any(r for l, r in history)

    return False


In [ ]:
violations = {}

def draw_violation(frame, trk, lane_info):
    x1, y1, x2, y2, tid = map(int, trk)

    if not lane_info["changed"]:
        return

    direction = lane_info["direction"]
    indicator_ok = indicator_used(tid, direction)
    is_violation = not indicator_ok

    if is_violation and tid not in violations:
        violations[tid] = {
            "vehicle_id": tid,
            "direction": direction,
            "indicator": "OFF"
        }
        cv2.imwrite(f"/content/evidence/vehicle_{tid}.jpg",
                    frame[y1:y2, x1:x2])

    color = (0,0,255) if is_violation else (0,255,0)

    cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
    cv2.putText(frame, f"ID {tid}", (x1, y1-60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    cv2.putText(frame, f"Lane Change: {direction}", (x1, y1-40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    cv2.putText(frame, f"Indicator: {'ON' if indicator_ok else 'OFF'}",
                (x1, y1-20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    cv2.putText(frame,
                "VIOLATION" if is_violation else "OK",
                (x1, y2+25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)


In [ ]:
# ============================================================
# LANE CHANGE VIOLATION DETECTION (REFINED & STABLE)
# ============================================================

import cv2
import numpy as np
from collections import deque

# ===================== CONFIGURATION =====================
OUTPUT_VIDEO = "/content/output_video/final_result.mp4"

# Lane change logic
LANE_HISTORY_LEN = 15
LANE_MOVE_RATIO = 0.06      # % of frame width

# Indicator logic
IND_HISTORY_LEN = 12
IND_MIN_ON_FRAMES = 4
IND_PIXEL_RATIO = 0.02

# HSV for yellow/orange indicator
IND_LOWER = np.array([5, 120, 120])
IND_UPPER = np.array([35, 255, 255])

# ===================== STATE STORAGE =====================
lane_centers = {}       # tid -> deque(cx)
indicator_states = {}   # tid -> deque(left,right)
final_decision = {}     # tid -> decision dict

# ===================== HELPER FUNCTIONS =====================
def compute_center(x1, x2):
    return int((x1 + x2) / 2)

def update_lane_centers(tid, cx):
    if tid not in lane_centers:
        lane_centers[tid] = deque(maxlen=LANE_HISTORY_LEN)
    lane_centers[tid].append(cx)

def detect_lane_change(tid, frame_width):
    """
    Returns: None / 'LEFT' / 'RIGHT'
    """
    if tid not in lane_centers:
        return None

    history = lane_centers[tid]
    if len(history) < LANE_HISTORY_LEN:
        return None

    delta = history[-1] - history[0]
    threshold = frame_width * LANE_MOVE_RATIO

    if delta > threshold:
        return "RIGHT"
    if delta < -threshold:
        return "LEFT"
    return None

def extract_indicator_rois(frame, box):
    x1, y1, x2, y2 = box
    w = x2 - x1
    h = y2 - y1
    y_start = y1 + int(0.7 * h)

    left_roi = frame[y_start:y2, x1:x1 + int(0.25 * w)]
    right_roi = frame[y_start:y2, x2 - int(0.25 * w):x2]

    return left_roi, right_roi

def indicator_is_on(roi):
    if roi.size == 0:
        return False
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, IND_LOWER, IND_UPPER)
    ratio = np.sum(mask > 0) / (roi.shape[0] * roi.shape[1] + 1e-6)
    return ratio > IND_PIXEL_RATIO

def update_indicator_states(frame, tracks):
    for x1, y1, x2, y2, tid in tracks:
        if tid not in indicator_states:
            indicator_states[tid] = deque(maxlen=IND_HISTORY_LEN)

        left_roi, right_roi = extract_indicator_rois(
            frame, (x1, y1, x2, y2)
        )
        indicator_states[tid].append(
            (indicator_is_on(left_roi), indicator_is_on(right_roi))
        )

def indicator_used_correctly(tid, direction):
    history = indicator_states.get(tid, [])
    if not history:
        return False

    if direction == "LEFT":
        count = sum(l for l, r in history)
    else:
        count = sum(r for l, r in history)

    return count >= IND_MIN_ON_FRAMES

# ===================== VIDEO SETUP =====================
cap = cv2.VideoCapture(INPUT_VIDEO)

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

# ===================== MAIN PROCESSING LOOP =====================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(
        frame,
        persist=True,
        conf=0.3,
        classes=[2, 3, 5, 7]   # vehicles only
    )

    if results[0].boxes.id is None:
        out.write(frame)
        continue

    boxes = results[0].boxes.xyxy.cpu().numpy()
    ids   = results[0].boxes.id.cpu().numpy()

    tracks = []
    for box, tid in zip(boxes, ids):
        x1, y1, x2, y2 = map(int, box)
        tracks.append([x1, y1, x2, y2, int(tid)])

    # 1️⃣ Update indicator history first
    update_indicator_states(frame, tracks)

    # 2️⃣ Process each tracked vehicle
    for x1, y1, x2, y2, tid in tracks:
        cx = compute_center(x1, x2)
        update_lane_centers(tid, cx)

        direction = detect_lane_change(tid, width)

        # Default visualization
        color = (0, 255, 0)
        label = f"ID {tid}"

        # Make decision ONCE
        if direction and tid not in final_decision:
            indicator_ok = indicator_used_correctly(tid, direction)
            final_decision[tid] = {
                "direction": direction,
                "indicator": indicator_ok
            }

        # If decision exists, draw full status
        if tid in final_decision:
            info = final_decision[tid]
            direction = info["direction"]
            indicator_ok = info["indicator"]

            if not indicator_ok:
                color = (0, 0, 255)
                label = f"ID {tid} | VIOLATION"
            else:
                label = f"ID {tid} | OK"

            cv2.putText(frame, f"Lane Change: {direction}",
                        (x1, y1 - 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            cv2.putText(frame, f"Indicator: {'ON' if indicator_ok else 'OFF'}",
                        (x1, y1 - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # Draw bounding box and ID
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label,
                    (x1, y2 + 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    out.write(frame)

cap.release()
out.release()

print("✅ FINAL OUTPUT VIDEO:", OUTPUT_VIDEO)
print("🚨 Total Violations:",
      sum(1 for v in final_decision.values() if not v["indicator"]))


In [ ]:
# ============================================================
# STEP 7: EVIDENCE CAPTURE + AUTO E-CHALLAN (CSV)
# ============================================================

import pandas as pd
import time
import os

# Create folders (safe check)
os.makedirs("/content/evidence", exist_ok=True)
os.makedirs("/content/challan", exist_ok=True)

challan_records = []

# Loop through final decisions
for tid, info in final_decision.items():
    if info["indicator"] is False:  # Violation only

        evidence_path = f"/content/evidence/vehicle_{tid}.jpg"

        # Save evidence frame (last seen frame)
        # We reuse the bounding box from last frame
        cap = cv2.VideoCapture(OUTPUT_VIDEO)
        cap.set(cv2.CAP_PROP_POS_FRAMES, cap.get(cv2.CAP_PROP_FRAME_COUNT) - 1)
        ret, frame = cap.read()
        cap.release()

        if ret:
            cv2.imwrite(evidence_path, frame)

        challan_records.append({
            "Vehicle_ID": tid,
            "Violation_Type": "Lane Change Without Indicator",
            "Direction": info["direction"],
            "Indicator_Status": "OFF",
            "Time": time.strftime("%Y-%m-%d %H:%M:%S"),
            "Evidence_Image": evidence_path
        })

# Save CSV
df = pd.DataFrame(challan_records)
csv_path = "/content/challan/violations.csv"
df.to_csv(csv_path, index=False)

print("✅ E-Challan generated successfully")
print("📄 CSV Path:", csv_path)
print("🖼 Evidence images saved in /content/evidence/")


In [ ]:
# ============================================================
# STEP 8: NUMBER PLATE RECOGNITION (ANPR - DEMO, EasyOCR)
# ============================================================

!pip install -q easyocr

import easyocr
import re
import cv2

# Initialize OCR reader (CPU only - stable for Colab)
reader = easyocr.Reader(['en'], gpu=False)

def extract_number_plate(vehicle_img):
    """
    Demo ANPR:
    Crops lower-middle vehicle region and applies OCR
    """

    if vehicle_img is None or vehicle_img.size == 0:
        return "UNKNOWN"

    h, w, _ = vehicle_img.shape

    # Plate ROI (bottom-middle)
    plate_roi = vehicle_img[
        int(0.6 * h):int(0.85 * h),
        int(0.2 * w):int(0.8 * w)
    ]

    if plate_roi.size == 0:
        return "UNKNOWN"

    # Preprocessing
    gray = cv2.cvtColor(plate_roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 11, 17, 17)
    _, thresh = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # OCR
    results = reader.readtext(thresh, detail=0)

    if not results:
        return "UNKNOWN"

    # Pick longest detected string
    plate = max(results, key=len)

    # Clean text
    plate = re.sub(r'[^A-Z0-9]', '', plate.upper())

    return plate if len(plate) >= 6 else "UNKNOWN"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# DEBUG: Show plates for ALL vehicles in one frame

# DEBUG: Show plates for ALL vehicles in ONE frame (SAFE VERSION)

cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    print("❌ Video not opened. Check INPUT_VIDEO path.")
else:
    ret, frame = cap.read()
    cap.release()

    if not ret or frame is None:
        print("❌ Could not read frame from video.")
    else:
        print("✅ Frame read successfully")

        # FORCE YOLO to use this frame
        results = model(frame, conf=0.3, classes=[2,3,5,7])

        if len(results) == 0 or results[0].boxes is None:
            print("❌ No vehicles detected")
        else:
            for box in results[0].boxes.xyxy:
                x1, y1, x2, y2 = box.cpu().numpy().astype(int)

                # Safety crop
                h, w, _ = frame.shape
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    print("Plate: INVALID CROP")
                    continue

                plate = extract_number_plate(crop)
                print("Plate:", plate)



In [ ]:
# ============================================================
# STEP 9A: GENERATE CSV E-CHALLAN
# ============================================================

import pandas as pd
import os

CSV_PATH = "/content/challan/violations.csv"
os.makedirs("/content/challan", exist_ok=True)

records = []

for tid, info in final_decision.items():
    if not info["indicator"]:  # only violations
        records.append({
            "Vehicle_ID": tid,
            "Direction": info["direction"],
            "Indicator": "OFF",
            "Plate": info.get("plate", "UNKNOWN")
        })

df = pd.DataFrame(records)
df.to_csv(CSV_PATH, index=False)

print("✅ CSV E-Challan Generated:", CSV_PATH)
print(df)


In [ ]:
!pip install reportlab
# ============================================================
# STEP 9B: GENERATE PDF E-CHALLAN
# ============================================================

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from datetime import datetime

PDF_PATH = "/content/challan/e_challan.pdf"

doc = SimpleDocTemplate(PDF_PATH, pagesize=A4)
styles = getSampleStyleSheet()
elements = []

elements.append(Paragraph("<b>E-CHALLAN REPORT</b>", styles["Title"]))
elements.append(Paragraph(
    f"Date & Time: {datetime.now().strftime('%d-%m-%Y %H:%M:%S')}",
    styles["Normal"]
))

# Table Data
table_data = [["Vehicle ID", "Direction", "Indicator", "Plate Number"]]

for _, row in df.iterrows():
    table_data.append([
        str(row["Vehicle_ID"]),
        row["Direction"],
        row["Indicator"],
        row["Plate"]
    ])

table = Table(table_data, colWidths=[80, 100, 100, 150])

table.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.grey),
    ("TEXTCOLOR", (0,0), (-1,0), colors.whitesmoke),
    ("GRID", (0,0), (-1,-1), 1, colors.black),
    ("ALIGN", (0,0), (-1,-1), "CENTER"),
    ("FONT", (0,0), (-1,0), "Helvetica-Bold")
]))

elements.append(table)
doc.build(elements)

print("✅ PDF E-Challan Generated:", PDF_PATH)
